# การเพิ่มประสิทธิภาพการรันโมเดลบนเครื่อง Local

นอกจากการอัพเกรด GPU เป็นตัวใหม่ๆแรงๆ ยังมีอีกหลายวิธีที่ทำให้สามารถใช้งาน LLMs ได้เร็วขึ้นบนเครื่องคอมพิวเตอร์ส่วนตัวได้ 

ใน Lab นี้จะพาไปสำรวจ 2 แนวทางหลักที่ช่วยให้การรันโมเดลลื่นไหลขึ้น

1. **การใช้ AI Framework ที่ปรับแต่งมาเพื่อ Hardware เฉพาะทาง (Hardware Optimization)**:

ในแล๊ปนี้รันบนเครื่อง Mac ซึ่งมีชิป Apple Silicon (M1/M2/M3) เราสามาระใช้ Library ที่ชื่อว่า MLX ซึ่งพัฒนาโดย Apple เพื่อดึงพลังของ Unified Memory และ GPU มาใช้ได้สูงสุด

_ข้อดี:_ ความเร็วสูงมาก (Inference speed) และติดตั้งง่ายสำหรับผู้ใช้ Mac

_ข้อเสีย:_ ผูกติดกับระบบปฏิบัติการ macOS และชิป Apple Silicon เท่านั้น

--------------

2. **การใช้โมเดลขนาดเล็ก หรือ เวอร์ชันที่ผ่านการบีบอัด (Quantized & Distilled Models)**: ยิ่งโมเดลมีขนาดเล็กลง ยิ่งสามารถรันบนเครื่องของเราได้เร็วขึ้น

_ข้อดี_: สามารถรันได้บนเครื่องเกือบทุกประเภท และประหยัดแรม (VRAM) อย่างเห็นได้ชัด

_ข้อเสีย_: ประสิทธิภาพโดยรวมอาจลดลงเมื่อเทียบกับโมเดลที่มีขนาดใหญ่กว่า

# MLX Ecosystem

In [2]:
# !uv pip install mlx-vlm torchvision

In [26]:
!python -m mlx_vlm generate --model mlx-community/Qwen3-VL-4B-Instruct-8bit --max-tokens 100 --temperature 0.0 --prompt "Describe this image." --image ./Examples/cats.jpg 

Fetching 14 files: 100%|█████████████████████| 14/14 [00:00<00:00, 30174.85it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Files: ['./Examples/cats.jpg'] 

Prompt: <|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>Describe this image.<|im_end|>
<|im_start|>assistant

This image features two cats sitting on a paved or stone surface outdoors, with a blurred background of what appears to be a wooden fence or structure and scattered leaves or debris on the ground.

The key visual element is the surreal and humorous contrast between the two cats:

- **The cat in the foreground** is a tabby cat with fur that has been digitally altered to appear entirely **green** — from its head, body, and tail to its paws. The green coloration is not natural and
Prompt: 534 tokens, 156.695 tokens-per-sec
Generation: 100 tokens, 19.476 tokens-per-sec
Peak memory: 5.950 GB


### MLX-VLM

`MLX-VLM` เป็นเครื่องมือใช้สำหรับโมเดลภาพ + ข้อความ.

_NOTE: ทำงานได้เร็วมาก แต่ใช้ได้กับแต่บางโมเดล_

### Run OpenAI-compatablity server

* run `python -m mlx_vlm server `
* check `http://0.0.0.0:8080/models`

In [39]:
import requests

def run_prompt(model, messages, stream=False):
    headers = {
        "Content-Type": "application/json"
    }

    data = {
        "model": model,
        "messages": messages,
        "stream": stream,
        "max_tokens": 500
        
    }
    return requests.post("http://localhost:8080/chat/completions", headers=headers, json=data)

In [42]:
start_time = time.time()

response = run_prompt("mlx-community/Qwen3-VL-4B-Instruct-8bit", [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Transcribe all text in this image exactly as it appears."
            },
            {
                "type": "input_image",
                "image_url": "./Examples/slip2.jpg"
            }
        ]
    }
])

end_time = time.time()
execution_time = end_time - start_time
print(f"Execution Time: {execution_time:.2f} sec")

responseContent = response.json()
responseContent

{'model': 'mlx-community/Qwen3-VL-4B-Instruct-8bit',
 'choices': [{'finish_reason': 'stop',
   'message': {'role': 'assistant',
    'content': 'CP ALL, 7-Eleven ปล. ประเวศ (08707)\nTAX#010754200011 (VAT Included)\nVat 06666 POS# :E02016000200345\nใบเสร็จรับเงิน/ใบกำกับภาษีสำเร็จ\n1 ปังเนยน้ำตาล 22.00\n1 แสตมป์ 3 บาท 0.00NP\nTotal (2) 22.00\nCash/Change 25.00 3.00\nR#0001192782 P2:0870782 26/08/59 09:13\n** ศูนย์ลูกค้าสัมพันธ์ 0-2711-7744 **'}}],
 'usage': {'input_tokens': 705,
  'output_tokens': 219,
  'total_tokens': 924,
  'prompt_tps': 183.7926452816227,
  'generation_tps': 16.66364500762707,
  'peak_memory': 6.022092089}}

In [43]:
print(responseContent["choices"][0]["message"]["content"])

CP ALL, 7-Eleven ปล. ประเวศ (08707)
TAX#010754200011 (VAT Included)
Vat 06666 POS# :E02016000200345
ใบเสร็จรับเงิน/ใบกำกับภาษีสำเร็จ
1 ปังเนยน้ำตาล 22.00
1 แสตมป์ 3 บาท 0.00NP
Total (2) 22.00
Cash/Change 25.00 3.00
R#0001192782 P2:0870782 26/08/59 09:13
** ศูนย์ลูกค้าสัมพันธ์ 0-2711-7744 **


In [48]:
response_stream = run_prompt("mlx-community/Qwen3-VL-4B-Instruct-8bit", [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Transcribe all text in this image exactly as it appears."
            },
            {
                "type": "input_image",
                "image_url": "./Examples/slip2.jpg"
            }
        ]
    }
], stream = True)


In [49]:
print("Assistant response:")
import json

for line in response_stream.iter_lines():
    # if line:
    decoded_line = line.decode('utf-8')
    if not decoded_line.startswith("data:"):
        if decoded_line.strip()=="":
            continue
            
        print(f"??", end="", flush=True)
        continue
        
    # Extract the JSON data after "data:"
    json_data = decoded_line[len("data:"):]
    try:
        event_data = json.loads(json_data)
        print(event_data["choices"][0]["delta"]["content"], end="", flush=True)
        
    except json.JSONDecodeError:
        print(f"Could not decode JSON: {json_data}")
    # else:
    #     print(line)
    
print()

Assistant response:
CP ALL, 7-Eleven ปล.ประเวศ(08707)
TAX#010754200011(VAT Included)
Vat 06666 POS# :E02016000200345
ใบเสร็จรับเงิน/ใบกำกับภาษีส่างชอ
1 ปังเนยน้าตาล 22.00
1 แสตมป์ 3 บาท 0.00NP
Total (2) 22.00 25.00 3.00
R#0001192782 P2:0870782 26/08/59 09:13
** ศูนย์อุปกรณ์สัมผั่นธ 0-2711-7744 **


### Using MLX-LM

`MLX-LM` เป็นเครื่องมือใช้สำหรับโมเดลข้อความเท่านั้น

In [97]:
# !uv pip install mlx-lm

In [51]:
from mlx_lm import load, generate

model, tokenizer = load("scb10x/typhoon-translate-4b-mlx-4bit")

# messages = [
#     {"role": "system", "content": "Translate the following text into English."},
#     {"role": "user", "content": "เมื่อสิ้นปี 2022 การเปิดตัว ChatGPT ของ OpenAI ถือเป็นจุดเปลี่ยนสำคัญ—โลกได้รู้จักกับ Generative AI (Gen AI) และทุกอย่างก็เปลี่ยนไป สิ่งที่เคยรู้สึกเหมือน frontier ไกลๆ กลายเป็นพลังในปัจจุบัน AI ถูกฝังตัวอย่างรวดเร็วทั้งในกิจวัตรส่วนตัวและกลยุทธ์องค์กร กลายเป็นตัวเปลี่ยนเกมอันทรงพลัง—ช่วยยกระดับชีวิต ปลดล็อกประสิทธิภาพใหม่ๆ และเปลี่ยนวิธีที่องค์กรดำเนินงาน ในการแสวงหาความได้เปรียบเชิงแข่งขัน ธุรกิจในหลากหลายภาคส่วนต่างเร่งทำความเข้าใจ นำไปใช้ และสร้างนวัตกรรมด้วย AI\nSCBX AI Outlook 2025: Beaconing the Future of Artificial Intelligence ถูกออกแบบมาเป็นประภาคารท่ามกลางคลื่นที่เร่งตัวขึ้นนี้—มอบความชัดเจนและทิศทางให้ผู้นำรับมือกับกระแสการเปลี่ยนแปลงทางเทคโนโลยี รายงานนี้สำรวจแนวโน้ม AI ที่กำลังกำหนดทิศทางในปีข้างหน้า และนำเสนอมุมมองเชิงกลยุทธ์ในการเปลี่ยนความไม่แน่นอนให้เป็นโอกาส รายงานแบ่งออกเป็นสี่ส่วน (Acts) แต่ละส่วนเน้นพลังสำคัญที่กำลังเปลี่ยนภูมิทัศน์ AI:\nACT I: Two Philosophies, One Future. The Battle Between Open-Source and Closed-Source AI Intensifies\nACT II: Tiny Titans - Small, but Mighty. More Versatile, Smaller, and Smarter: 3 Trends of the Next AI Evolution\nACT III: AI at Your Fingertips. Agentic AI: Rise of the Agents\nACT IV: Not Quite Human, But Almost There. Artificial General Intelligence (AGI) and the Unresolved Path to Human-Level AI\nรายงานปิดท้ายด้วย EPILOGUE: The AI Storm – Infinite Impact พร้อม Case Studies จากภายในดวงตาของพายุไต้ฝุ่น—นำเสนอกรณีศึกษาจริงจาก SCBX ในการใช้ AI Engine “Typhoon” ของกลุ่มในหน่วยธุรกิจต่างๆ ขณะที่กระแส AI กำลังก้าวไปข้างหน้า รายงานนี้จึงไม่ได้เป็นเพียงการคาดการณ์ แต่เป็นประภาคารเชิงกลยุทธ์สำหรับผู้พร้อมจะก้าวขึ้นไปกับคลื่นและเป็นผู้นำในอนาคต"},
# ]

text = '''
What is AI? 
Artificial intelligence (AI) is technology that enables computers and machines to simulate human learning, comprehension, problem solving, decision making, creativity and autonomy.

Applications and devices equipped with AI can see and identify objects. They can understand and respond to human language. They can learn from new information and experience. They can make detailed recommendations to users and experts. They can act independently, replacing the need for human intelligence or intervention (a classic example being a self-driving car).
'''

messages = [
    {"role": "system", "content": "Translate the following text into Thai."},
    {"role": "user", "content": text},
]

prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

response = generate(model, tokenizer, prompt=prompt, verbose=True, max_tokens=2048)


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

ปัญญาประดิษฐ์ (AI) คืออะไร?  
ปัญญาประดิษฐ์ (AI) คือเทคโนโลยีที่ช่วยให้คอมพิวเตอร์และเครื่องจักรสามารถจำลองการเรียนรู้ การเข้าใจ การแก้ปัญหา การตัดสินใจ ความคิดสร้างสรรค์ และการทำงานอย่างเป็นอิสระของมนุษย์ได้  

อุปกรณ์และแอปพลิเคชันที่ใช้ AI สามารถมองเห็นและระบุวัตถุต่างๆ ได้ พวกมันสามารถเข้าใจและตอบสนองต่อภาษาของมนุษย์ได้ พวกมันสามารถเรียนรู้จากข้อมูลใหม่และประสบการณ์ที่ได้รับ และสามารถให้คำแนะนำที่ละเอียดแก่ผู้ใช้และผู้เชี่ยวชาญได้ นอกจากนี้ยังสามารถทำงานได้อย่างอิสระ โดยไม่จำเป็นต้องอาศัยการควบคุมหรือแทรกแซงจากมนุษย์ (ตัวอย่างที่ชัดเจนคือรถยนต์ไร้คนขับ)
Prompt: 117 tokens, 117.346 tokens-per-sec
Generation: 177 tokens, 36.683 tokens-per-sec
Peak memory: 2.702 GB


### MLX-Audio

**Try this**
```
mlx_audio.tts.generate --text "I have a dream that my four little children will one day live in a nation where they will not be judged by the color of their skin but by the content of their character. I have a dream today" --model mlx-community/Kokoro-82M-bf16  --lang_code a
```

In [106]:
# !uv pip install mlx-audio
# !uv pip install mlx-whisper

# brew install ffmpeg

**Try this**

```
mlx_whisper Examples/audio.mp3 --language th --model tawankri/distill-thonburian-whisper-large-v3-mlx 
```

# Optimised Models

#### `Distilled` != `Quantized` models

* `distilled` = โมเดลขนาดเล็กที่ถูกเทรนโดยให้ลอกเลียนแบบโมเดลขนาดใหญ่เพื่อถ่ายทอดความรู้ (train a small model using a larger model as a teacher)
* `quantized` = โมเดลขนาดเล็กจากเทคนิคกการลดความละเอียดของพารามิเตอร์ภายในโมเดลเพื่อให้โมเดลที่มีอยู่เดิมมีขนาดเล็กลง (reduce the precision of an existing model's parameters)


เลือกดูโมเดลได้ที่ https://huggingface.co/models

* ใช้คีย์เวิร์ด `gguf` ในการค้นหาสำหรับโมเดลประเภท quantized
* ใช้คีย์เวิร์ด `mlx` ในการค้นหาสำหรับโมเดลที่รองรับ MLX


#### IMPORTANTLLY, 
โดยปกติแล้วโมเดลที่ตัวเล็กกว่าจะ **ทำงานได้เร็วกว่า** แต่จะมี **ความแม่นยำน้อยลง** !!

In [1]:
import ollama
from datetime import datetime
import time


## ตัวอย่าง Translation task

In [2]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import ollama
import pandas as pd

def get_wiki_content(url, n_paragraph = 4):
    headers = {'User-Agent': 'SilpakornLLMCourse/1.0'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    paragraphs = soup.find_all('p')
    content = "\n\n".join([p.get_text() for p in paragraphs[0:n_paragraph]]) 
    return content

wiki_url = "https://en.wikipedia.org/wiki/Silpakorn_University"
source_text = get_wiki_content(wiki_url)

print(f"ดึงข้อมูลสำเร็จ: {len(source_text)} ตัวอักษร\n")

ดึงข้อมูลสำเร็จ: 2642 ตัวอักษร



#### ผลลัพท์จาก Original Model

In [75]:
start_time = datetime.now()
print(f"[{start_time.strftime('%H:%M:%S')}]")

prompt = 'You are a professional translator. Translate the following English text into Thai. Keep the original meaning and use proper Thai academic tone. Output as Markdown.'
response = ollama.chat(
	model="scb10x/typhoon2.5-qwen3-4b",
    messages=[
		{
			'role': 'system',
			'content': prompt
		},
        {
			'role': 'user',
			'content': f"Translate this from English into Thai: \n\n{source_text}"
		},
	],
    options={
        # "temperature": 0
    }
)

end_time = datetime.now()
duration = end_time - start_time

print(f"[{end_time.strftime('%H:%M:%S')}] **Processed** in {duration.total_seconds():.3f} seconds.")

[16:42:23]
[16:43:18] **Processed** in 55.457 seconds.


In [79]:
from IPython.display import display, Markdown

text = response.message.content
display(Markdown(text))

มหาวิทยาลัยศิลปากร (SU.) (ภาษาไทย: มหาวิทยาลัยศิลปากร; RTGS: Mahawitthayalai Sinlapakon; ก็มีชื่อเรียกอีกอย่างว่า มหาวิทยาลัยศิลปะแห่งประเทศไทย) เป็นมหาวิทยาลัยแห่งชาติหนึ่งในประเทศไทย โดยมหาวิทยาลัยแห่งนี้ก่อตั้งขึ้นในกรุงเทพมหานครปี พ.ศ. 2466 โดยศาสตราจารย์ศิลปะชาวทัสคานีนาม คอร์ราโด เฟโรชี ซึ่งเมื่อได้รับสัญชาติไทยแล้วเปลี่ยนชื่อไทยเป็น สิลปา เบื้องรัช มหาวิทยาลัยแห่งนี้เริ่มต้นมาเป็นสถาบันศิลปะและปัจจุบันได้ขยายกิจกรรมไปสู่คณะอื่นๆ อีกหลายคณะ เมื่อปี พ.ศ. 2559 มีนักศึกษาทั้งหมดจำนวน 25,210 คน[4]

มหาวิทยาลัยศิลปากรได้ก่อตั้งขึ้นตั้งแต่ปี พ.ศ. 2466 เป็นโรงเรียนศิลปะภายใต้กรมศิลปากรของประเทศไทย โดยมีหลักสูตรศิลปะเฉพาะทางด้านการวาดภาพและการตัดต่อประติมากรรมเท่านั้น และยกเว้นค่าเล่าเรียนสำหรับเจ้าหน้าที่รัฐและนักเรียน โดยการก่อตั้งโรงเรียนศิลปะนี้มีอิทธิพลมากจากความตั้งใจทำงานศิลปะอย่างต่อเนื่องของศาสตราจารย์สิลปา เบื้องรัช ศิลปินชาวอิตาลี (เดิมชื่อคอร์ราโด เฟโรชี) ที่ได้รับเชิญให้ทำงานในกรมศิลปากรในสมัยสมเด็จพระราชาธิบดีที่ 6 และภายหลังได้ขยายการสอนไปสู่กลุ่มประชาชนทั่วไปมากยิ่งขึ้น ก่อนจะก่อตั้งโรงเรียนศิลปะขึ้นมา โรงเรียนศิลปะดังกล่าวได้พัฒนาขึ้นอย่างค่อยเป็นค่อยไป และในวันที่ 12 ตุลาคม พ.ศ. 2466 ได้รับการประกาศสถาปนาเป็นมหาวิทยาลัยอย่างเป็นทางการภายใต้ชื่อว่า “มหาวิทยาลัยศิลปากร” โดยคณะแรกที่ก่อตั้งขึ้นคือคณะศิลปะการวาดภาพและการตัดต่อประติมากรรม ในปี พ.ศ. 2498 ก่อตั้งคณะสถาปัตยกรรมไทย ซึ่งภายหลังได้เปลี่ยนชื่อเป็นคณะสถาปัตยกรรม และก่อตั้งคณะอีกสองคณะคือคณะโบราณคดี และคณะศิลปะประดิษฐ์

ในปี พ.ศ. 2493 มหาวิทยาลัยศิลปากรได้ขยายกิจกรรมให้ครอบคลุมสาขาต่างๆ โดยแบ่งคณะต่างๆ ออกเป็นสาขาเฉพาะทางเพื่อเพิ่มความหลากหลายของหลักสูตร แต่สถานที่หลักในเมืองวังธาราฟรา กลับไม่เพียงพอสำหรับการขยายตัวนี้ จึงได้ก่อตั้งสถานศึกษาใหม่แห่งหนึ่งที่ชื่อว่า “วังสรรค์ชันดร์” ในจังหวัดนครปฐม ซึ่งตั้งอยู่ในพื้นที่เดิมที่เคยเป็นที่พักอาศัยของสมเด็จพระราชาธิบดีที่ 6 คณะแรกที่ก่อตั้งขึ้นบนสถานศึกษานี้คือคณะศิลปะในปี พ.ศ. 2495 และคณะศึกษาศาสตร์ในปี พ.ศ. 2497 ต่อมาได้ก่อตั้งคณะอีกสามคณะ ได้แก่ คณะวิทยาศาสตร์ในปี พ.ศ. 2495 คณะเภสัชศาสตร์ในปี พ.ศ. 2519 และคณะวิศวกรรมศาสตร์และเทคโนโลยีอุตสาหกรรมในปี พ.ศ. 2525 ในปี พ.ศ. 2532 ก่อตั้งคณะดนตรีขึ้นมา

ในปี พ.ศ. 2530 มหาวิทยาลัยศิลปากรได้ขยายขอบเขตการศึกษาออกไปยังจังหวัดเพชรบุรี โดยก่อตั้งสถานศึกษาใหม่ชื่อว่า “จังหวัดเพชรบุรี ศูนย์เทคโนโลยีสารสนเทศ” ในปี พ.ศ. 2534 ก่อตั้งคณะวิทยาศาสตร์สัตว์และเทคโนโลยีการเกษตร และคณะวิทยาศาสตร์บริหารจัดการขึ้นบนสถานศึกษาดังกล่าว ในปี พ.ศ. 2536 ก่อตั้งคณะเทคโนโลยีสารสนเทศและการสื่อสาร (ICT) และมหาวิทยาลัยศิลปากรนานาชาติ (SUIC) โดยมีหน้าที่ให้ส่งเสริมหลักสูตรระดับนานาชาติในสาขาต่างๆ ที่ไม่ได้อยู่ในหลักสูตรเดิม

#### ผลลัพท์จาก Quantised Model

In [89]:
from ollama import Client

ollama = Client(host='http://localhost:11434')
start_time = datetime.now()
print(f"[{start_time.strftime('%H:%M:%S')}]")

response = ollama.chat(
	model="hf.co/typhoon-ai/typhoon2.5-qwen3-4b-gguf:Q4_K_M",
    # model="hf.co/mradermacher/typhoon2.5-qwen3-4b-GGUF:Q4_K_M",
    messages=[
		{
			'role': 'system',
			'content': prompt
		},
        {
			'role': 'user',
			'content': f"Translate this from English into Thai: \n\n{source_text}"
		},
	],
    options={
        # "temperature": 0
    }
)

end_time = datetime.now()
duration = end_time - start_time

print(f"[{end_time.strftime('%H:%M:%S')}] **Processed** in {duration.total_seconds():.3f} seconds.")

[19:08:01]
[19:11:26] **Processed** in 205.156 seconds.


In [90]:
prompt

'You are a professional translator. Translate the following English text into Thai. Keep the original meaning and use proper Thai academic tone. Output as Markdown.'

In [91]:
from IPython.display import display, Markdown

text = response.message.content
display(Markdown(text))

มหาวิทยาลัยศิลปากร (SU.) (ไทย: มหาวิทยาลัยศิลปากร; RTGS: Mahawitthayalai Sinlapakon; ก็มีชื่อเรียกอีกอย่างว่า มหาวิทยาลัยศิลปะการแพทย์แห่งประเทศไทย) เป็นมหาวิทยาลัยแห่งชาติของประเทศไทย มีการก่อตั้งขึ้นในกรุงเทพฯ ในปี พ.ศ. 2486 โดยศาสตราจารย์ศิลปินชาวอิตาลีคนหนึ่งชื่อ คอร์รัลดอร์ เฟอโรชี ซึ่งเมื่อได้รับสัญชาติไทยแล้วก็เปลี่ยนชื่อเป็นซิลปา เบียรัสซี่ มหาวิทยาลัยแห่งนี้ตั้งต้นมาเป็นมหาวิทยาลัยศิลปะ และปัจจุบันก็ขยายกิจกรรมเข้าสู่สาขาอื่นๆ อีกหลายสาขาในปี พ.ศ. 2559 มีนักศึกษาทั้งหมดจำนวน 25,210 คน

มหาวิทยาลัยศิลปากรก่อตั้งขึ้นตั้งแต่ปี พ.ศ. 2496 ในหน่วยงานศิลปะแห่งชาติของประเทศไทย เป็นสถาบันที่จัดการศึกษาเฉพาะด้านศิลปะเฉพาะเช่น การวาดภาพและการประดิษฐ์งานศิลป์ โดยยกเว้นค่าเทอมสำหรับเจ้าหน้าที่รัฐและนักศึกษา โดยการก่อตั้งมหาวิทยาลัยแห่งนี้ได้รับอิทธิพลจากความตั้งใจอันยาวนานของศาสตราจารย์ซิลปา เบียรัสซี่ ศิลปินชาวอิตาลีคนหนึ่ง (เคยใช้ชื่อว่า คอร์รัลดอร์ เฟอโรชี) ซึ่งได้รับมอบหมายให้ทำงานในหน่วยงานศิลปะแห่งชาติในสมัยสมเด็จพระบาทสมเด็จพระรัชกาลที่หก เขาขยายการสอนออกไปเพื่อรวมนักศึกษาทั่วไปเข้ามา และก่อตั้งโรงเรียนศิลปะขึ้นมา จากนั้นโรงเรียนแห่งนี้ก็พัฒนาต่อเนื่องจนได้รับสถานะใหม่เป็นมหาวิทยาลัยอย่างเป็นทางการ และได้รับชื่อว่า “มหาวิทยาลัยศิลปากร” ในวันที่ 12 ตุลาคม พ.ศ. 2486 มีคณะแรกเปิดสอนคือ คณะศิลปะและสถาปัตยกรรม จากนั้นในปี พ.ศ. 2498 ก็จัดตั้งคณะสถาปัตยกรรมขึ้นมา แล้วจัดตั้งคณะโบราณคดี และคณะศิลปะประดิษฐ์ขึ้นมาอีกสองคณะ

ในปี พ.ศ. 2539 มหาวิทยาลัยศิลปากรได้ขยายสาขาต่างๆ ออกเป็นสาขาย่อย เพื่อขยายบริการและกิจกรรมให้หลากหลายขึ้น แต่สถานที่ในวังธาราฟราของมหาวิทยาลัยยังไม่สามารถรองรับความต้องการได้ จึงได้ก่อตั้งสถานศึกษาใหม่ชื่อว่า พิพิธภัณฑ์สานต์ชานะ ซึ่งตั้งอยู่ในจังหวัดนครปฐม ซึ่งเป็นพื้นที่เดิมที่สมเด็จพระบาทสมเด็จพระรัชกาลที่หกมีที่พำนักอาศัย คณะแรกที่จัดตั้งขึ้นบนสถานศึกษานี้คือ คณะศิลปะ ในปี พ.ศ. 2515 และคณะศึกษาศาสตร์ ในปี พ.ศ. 2523 จากนั้นจึงได้จัดตั้งคณะอีกสามคณะขึ้นมา ได้แก่ คณะวิทยาศาสตร์ ในปี พ.ศ. 2525 คณะเภสัชศาสตร์ ในปี พ.ศ. 2549 และคณะวิศวกรรมศาสตร์และเทคโนโลยีอุตสาหกรรมในปี พ.ศ. 2555 ในปี พ.ศ. 2552 ก็จัดตั้งคณะดนตรีขึ้นมาได้

ในปี พ.ศ. 2540 มหาวิทยาลัยศิลปากรก็ขยายตัวออกไปอีกครั้งโดยก่อตั้งสถานศึกษาใหม่ในจังหวัดเพชรบุรี โดยสถานศึกษาแห่งนี้ชื่อว่า “จังหวัดเพชรบุรีสำนักงานเทคโนโลยีสารสนเทศ” ในปี พ.ศ. 2534 และ พ.ศ. 2535 ก็ได้จัดตั้งคณะวิทยาศาสตร์สัตว์และเทคโนโลยีการเกษตร และคณะวิทยาศาสตร์บริหารศาสตร์ขึ้นมาบนสถานศึกษาแห่งนี้ ในปี พ.ศ. 2536 ก็ได้จัดตั้งคณะเทคโนโลยีสารสนเทศและการสื่อสารขึ้นมา และได้ก่อตั้งมหาวิทยาลัยศิลปากรนานาชาติ (SUIC) อีกด้วย โดยหน้าที่ของมหาวิทยาลัยศิลปากรนานาชาตินี้คือให้บริการหลักสูตรนานาชาติในสาขางานศึกษาอื่นๆ อีกด้วยครับ

*Note: Some facts have been revised and corrected based on historical accuracy and official sources. The original English version is referenced for context.*

In [176]:
# text = json.loads(response.message.content)["natural_text"]
# print(text)